In [1]:
import torch

In [2]:
x = torch.tensor(3.0, requires_grad=True) 

In [3]:
y = x**2

In [4]:
x

tensor(3., requires_grad=True)

In [5]:
y # grad_fn -> previous function 

tensor(9., grad_fn=<PowBackward0>)

In [6]:
y.backward() # gradients get calculated

In [7]:
x.grad

tensor(6.)

In [8]:
# example 2
import math

def dz_dx(x):
    return 2*x*math.cos(x**2)

In [18]:
dz_dx(4)

-7.661275842587077

In [19]:
# computing using pytorch
x = torch.tensor(4.0, requires_grad=True)

In [20]:
y = x ** 2

In [21]:
z = torch.sin(y)

In [22]:
x

tensor(4., requires_grad=True)

In [23]:
y

tensor(16., grad_fn=<PowBackward0>)

In [24]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [25]:
z.backward() # gradients calculated

In [26]:
x.grad 

tensor(-7.6613)

In [27]:
y.grad # not a leaf node hence gradient will not be calculated

/var/folders/yz/s57wsyt918j04kt6btpgwgy40000gn/T/ipykernel_77172/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


In [28]:
# input tensor -> leaf
# output tensor -> root

In [29]:
# inputs
x = torch.tensor(6.7) # input feature
y = torch.tensor(0.0) # true label (binary)

w = torch.tensor(1.0) # weight
b = torch.tensor(0.0) # bias

In [30]:
# binary cross entropy for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8 # to prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1-epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [31]:
# forward pass
z = w * x + b # weighted sum (linear part)
y_pred = torch.sigmoid(z) # predicted probability

# compute binary cross entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [32]:
loss

tensor(6.7012)

In [33]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [34]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [35]:
# now using autograd -> very few lines of code
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [36]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [37]:
w

tensor(1., requires_grad=True)

In [38]:
b

tensor(0., requires_grad=True)

In [39]:
# forward propagation
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [40]:
# actication
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [41]:
# calculating loss
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [42]:
loss.backward() # done

In [43]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [44]:
# input as vector
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [45]:
x

tensor([1., 2., 3.], requires_grad=True)

In [46]:
y = (x ** 2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [47]:
y.backward()

In [48]:
x.grad # gradient for each dimension

tensor([0.6667, 1.3333, 2.0000])

In [49]:
# clearing gradients
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [61]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [62]:
y.backward()

In [63]:
x.grad

tensor(4.)

In [60]:
x.grad.zero_() # clearing gradients

tensor(0.)

In [64]:
# disable gradient tracking -> during prediction
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [65]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [66]:
y.backward()

In [67]:
x.grad

tensor(4.)

In [68]:
# now training is complete
# only forward pass for prediction
# no need of backward tracking

In [69]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [70]:
x.requires_grad_(False)

tensor(2.)

In [71]:
x 

tensor(2.)

In [72]:
y = x ** 2

In [73]:
y

tensor(4.)

In [74]:
y.backward() # will return error

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [75]:
# option 2 -> using detach()
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [76]:
z = x.detach()
z # exactly same values as x, but detach from computational graph and gradient tracking is not on

tensor(2.)

In [77]:
y = x ** 2

In [78]:
y

tensor(4., grad_fn=<PowBackward0>)

In [81]:
y1 = z ** 2

In [82]:
y1

tensor(4.)

In [83]:
y.backward()

In [84]:
y1.backward() # error

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [85]:
# option 3 -> more convinient
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [86]:
with torch.no_grad():
    y = x ** 2 # no tracking

In [87]:
y

tensor(4.)

In [88]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn